# 🧭 Meta-Controllers: From Specialist Routing to Architecture Selection

A **router** reads each request and dispatches it to one specialist. A **meta-controller** makes the same kind of
decision one level up: it reads a task and chooses which whole **architecture** should handle it — a single call, a
search agent, a multi-step planner or a debate. This notebook builds the router first, as a LangGraph graph, reuses
its search agent as one of the meta-controller's strategies, and checks both controllers' choices in one table.

## Learning Objectives
In this notebook, you will learn:
1. **Routing as a graph** - A typed (`Literal`) decision dispatches each request to one specialist node with `add_conditional_edges`
2. **Tool-using specialists** - Build the search specialist with `create_agent`, so its search tool actually runs
3. **Strategies as callables** - Put four whole architectures behind one `(query) -> str` signature, reusing Part 1's search agent
4. **Classifier and registry** - A `Strategy` Enum decision and a registry dict that maps each strategy to its callable
5. **Checked routing** - One code-computed table of expected vs chosen routes for both controllers

## Where this fits
The blackboard notebooks (`01_blackboard_intro.ipynb`, `02_blackboard_research_team.ipynb`) let a controller
*sequence* several specialists on one task. A router decides **once** per request and steps aside, like module 06's
supervisor without the loop. That is also how it differs from a handoff
([`01_agent_handoffs.ipynb`](../../m25_multi_agent_architectures/notebooks/05a_agent_handoffs.ipynb), module 07), where the agent holding the conversation decides to pass
it on. `06_ensemble_parallel_exploration.ipynb` then runs several agents on the same task at once.

## Prerequisites
- `create_agent` agents and typed routing decisions (modules 05–07)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `TAVILY_API_KEY` in the same `.env` (web search for the Researcher, which Part 2 reuses as the search-agent strategy)

| Property | Value |
|---|---|
| Origin | The "Meta-Controller" notebook (a router over architectures) in the community `all-agentic-architectures` collection (FareedKhan-dev). Routing in front of specialists is older: LangChain's legacy `RouterChain` did it, and a LangGraph conditional edge does it today |

### Definition
A **meta-controller** is a classify-and-dispatch layer whose decision is *which agentic architecture to run* for a
task. A **router** (Part 1) makes the same kind of decision inside one fixed architecture: *which specialist*
answers. Both look at the request once, pick one option from a fixed set with a stated reason, and hand over. Neither
sequences several steps on one task; that is what a blackboard controller does. The summary compares the three.

### High-level Workflow

1. **Classify:** One LLM call receives the request and a description of each option, and returns a structured
   decision: the option (one of a fixed set) and the reasoning.
2. **Dispatch:** Code looks up what is registered for that option: a graph node for the router, a callable for the
   meta-controller.
3. **Execute:** The chosen specialist or architecture runs on its own terms (a single call, a tool loop, a
   plan/answer/synthesize sequence or a two-stance debate), with its own control flow, invisible to the controller.
4. **Return:** The output comes back tagged with the choice and its reason, so every decision stays inspectable.

### When to Use / Applications
* **General-purpose assistants** that must handle everything from trivia to research to judgment calls without forcing one workflow on every query.
* **Cost/latency-sensitive systems**: simple questions get one cheap call; only the tasks that need it pay for a tool loop or a multi-agent pattern.
* **Growing platforms**: a new specialist or strategy is one more registered entry and one more line in the classifier prompt.

### Strengths & Weaknesses
* **Strengths:**
  * **Right-sized cost:** simple requests stay cheap; only live-data or complex tasks pay for the expensive options.
  * **Composability:** each option is self-contained and testable alone; the controller only needs to know *when* to pick it.
  * **Inspectable decisions:** the `reasoning` field makes every choice auditable, which matters when the wrong option would still produce a plausible-looking answer (a single-shot guess at a live-data question).
* **Weaknesses:**
  * **Classification errors cascade:** a misjudged request goes to the wrong option, and this minimal design has no fallback.
  * **Coarse granularity:** one choice per task cannot handle a genuine mixture ("look up X, then debate whether it is a good idea").
  * **Extra latency:** one LLM call is spent on deciding before any real work starts.

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads `OPENAI_API_KEY` and `TAVILY_API_KEY`, and sets two caps:
the steps each `create_agent` loop may take, and the number of sub-questions the planning strategy may answer.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys, imports and the cost caps
# ============================================================================
import warnings
from enum import Enum
from typing import Any, Dict, List, Literal, Optional

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown
from typing_extensions import TypedDict

console = Console()

AGENT_RECURSION_LIMIT = 12  # graph steps each create_agent loop may take (model and tool calls)
MAX_SUB_STEPS = 4           # sub-questions the multi-step planning strategy answers at most (Part 2)

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by both controllers, the specialists and every strategy
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔀 Part 1: Ordinary Routing as a Graph

The router reads one request, picks one specialist with a typed decision, and the graph ends after that specialist
replies. The pool has three members:
1. **Generalist:** casual conversation and simple questions
2. **Researcher:** a search-enabled agent for recent events and facts it must look up
3. **Coder:** writes Python code

**Routing vs handoff.** Here the *router* decides, before any specialist runs, and a specialist cannot pass the
request on. In a handoff ([`01_agent_handoffs.ipynb`](../../m25_multi_agent_architectures/notebooks/05a_agent_handoffs.ipynb)) the agent that holds the conversation decides,
by calling a transfer tool, and the next turn starts with whoever holds it then.

### 1.1 👥 The Specialists
Each specialist is a `create_agent` agent. The Researcher's agent gets the Tavily search tool, and `create_agent`
**runs** it in a loop until the model writes its answer. (Binding a tool with `llm.bind_tools()` alone only lets the
model *request* a search; nothing runs it, and the reply is often an empty message holding the tool call.) `ask()`
runs any of these agents on one request and also counts the tool calls it made, so the Researcher's searches are
measured rather than assumed.

In [ ]:
# ============================================================================
# SPECIALIST AGENTS: three create_agent agents, one of them with web search
# ============================================================================
search_tool = TavilySearch(max_results=3)


def ask(agent, request: str) -> tuple[str, int]:
    """Run a create_agent agent on one request; return its answer and the number of tool calls it made."""
    result = agent.invoke({"messages": [HumanMessage(content=request)]}, {"recursion_limit": AGENT_RECURSION_LIMIT})
    tool_calls = sum(len(getattr(message, "tool_calls", None) or []) for message in result["messages"])
    return result["messages"][-1].content, tool_calls


def make_specialist(persona: str, tools: Optional[list] = None):
    """A specialist is a create_agent agent; tools are optional."""
    system_prompt = f"You are a specialist agent. {persona} Respond directly and concisely to the user's request."
    return create_agent(llm, tools=tools or [], system_prompt=system_prompt)


SPECIALIST_AGENTS = {
    "Generalist": make_specialist(
        "You are a friendly and helpful generalist AI assistant. You handle casual conversation and simple questions."
    ),
    "Researcher": make_specialist(
        "You are an expert researcher. Use your search tool to find current information before answering.",
        tools=[search_tool],
    ),
    "Coder": make_specialist(
        "You are an expert Python programmer. Provide only clean, efficient code in a markdown code block, "
        "with minimal explanation."
    ),
}


class MetaAgentState(TypedDict):
    user_request: str
    next_agent_to_call: Optional[str]
    routing_reason: Optional[str]
    generation: str
    tool_calls: int


def specialist_node(name: str):
    """Graph node that runs one specialist agent on the request."""

    def node(state: MetaAgentState) -> Dict[str, Any]:
        answer, tool_calls = ask(SPECIALIST_AGENTS[name], state["user_request"])
        return {"generation": answer, "tool_calls": tool_calls}

    return node


print(f"✅ Specialists ready: {list(SPECIALIST_AGENTS)} (only the Researcher has a tool: {search_tool.name})")

### 1.2 🧠 The Router

The router's only job is to choose a specialist. Its decision schema uses a `Literal`, so the answer is always a valid
node name, and the reason is kept in state for inspection.

In [ ]:
# ============================================================================
# ROUTER NODE: one typed decision per request
# ============================================================================
SPECIALISTS = {
    "Generalist": "Handles casual conversation, greetings, and simple questions.",
    "Researcher": "Answers questions about recent events, complex topics, or anything requiring up-to-date information from the web.",
    "Coder": "Writes Python code based on a user's specification.",
}


class RouteDecision(BaseModel):
    next_agent: Literal["Generalist", "Researcher", "Coder"] = Field(description="The specialist to call.")
    reasoning: str = Field(description="A brief reason for choosing this specialist.")


router_llm = llm.with_structured_output(RouteDecision)


def router_node(state: MetaAgentState) -> Dict[str, Any]:
    """Decide which specialist handles the request."""
    specialist_descriptions = "\n".join(f"- {name}: {desc}" for name, desc in SPECIALISTS.items())
    decision = router_llm.invoke(
        f"""You are the router for a multi-agent AI system. Route the user's request to the most appropriate specialist.

Available specialists:
{specialist_descriptions}

User Request: "{state['user_request']}"
"""
    )
    console.print(f"[yellow]Routing decision:[/yellow] {decision.next_agent} ([italic]{decision.reasoning}[/italic])")
    return {"next_agent_to_call": decision.next_agent, "routing_reason": decision.reasoning}


print("✅ Router defined")

### 1.3 🏗️ Graph Dispatch

The router is the entry point. A conditional edge sends the state to the chosen specialist, and the graph ends after
that specialist replies. A pure routing function reads the decision from state; the LLM call stays in the node.

In [ ]:
# ============================================================================
# ASSEMBLE THE ROUTER GRAPH
# ============================================================================
def route_to_specialist(state: MetaAgentState) -> str:
    return state["next_agent_to_call"]


workflow = StateGraph(MetaAgentState)
workflow.add_node("router", router_node)
for name in SPECIALISTS:
    workflow.add_node(name, specialist_node(name))

workflow.add_edge(START, "router")
workflow.add_conditional_edges("router", route_to_specialist, {name: name for name in SPECIALISTS})
for name in SPECIALISTS:
    workflow.add_edge(name, END)

router_app = workflow.compile()
print("✅ Router graph compiled. Edges:")
for edge in router_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### 1.4 ▶️ Three Requests

Three requests, each written for a different specialist. The expected specialist is recorded next to each request,
so Part 4 can check the routes in code. The tool-call count after each answer shows whether the Researcher really
searched.

In [ ]:
# ============================================================================
# RUN THREE REQUESTS THROUGH THE ROUTER
# ============================================================================
router_cases = [
    ("Hello, how are you today?", "Generalist"),
    ("What is the latest stable release of Python, and when was it released?", "Researcher"),
    ("Can you write me a python function to calculate the nth fibonacci number?", "Coder"),
]

router_results = []
for request, expected in router_cases:
    console.print(f"\n[bold]Request:[/bold] {request}")
    result = router_app.invoke({"user_request": request})
    console.print(Markdown(result["generation"]))
    print(f"-> answered by {result['next_agent_to_call']} with {result['tool_calls']} tool call(s)")
    router_results.append((request, expected, result["next_agent_to_call"], result["tool_calls"]))

---
## 🏛️ Part 2: Strategies as Callables

Part 1's router chose *who* answers inside one fixed graph. The meta-controller chooses *how*: which whole
architecture handles the task. Each architecture is wrapped as a plain callable `(query: str) -> str`, so the
controller never needs to know what happens inside. These are compact stand-ins for fuller builds (the debate in
`03_debate.ipynb`, for example); the point of this notebook is the selection logic:

1. **`run_direct_answer`**: a single LLM call. Best for simple, self-contained factual questions the model already knows.
2. **`run_search_agent`**: Part 1's Researcher, the `create_agent` loop with Tavily search. Best for questions that need live or external information.
3. **`run_multi_step_planning`**: splits the task into sub-questions (at most `MAX_SUB_STEPS`), answers each, then synthesizes. Best for multi-part questions.
4. **`run_debate`**: a "for" stance, an "against" stance, then a judge's balanced verdict. Best for opinion and trade-off questions with no single factual answer.

In [ ]:
# ============================================================================
# STRATEGY 1: DIRECT ANSWER
# ============================================================================
def run_direct_answer(query: str) -> str:
    """Single LLM call — for simple, self-contained factual questions."""
    console.print("--- STRATEGY [direct_answer]: Answering in a single call... ---")
    response = llm.invoke(
        f"Answer the following question directly and concisely.\n\nQuestion: {query}"
    )
    return response.content

print("✅ run_direct_answer() defined: 1 LLM call")

In [ ]:
# ============================================================================
# STRATEGY 2: SEARCH AGENT (Part 1's Researcher, a create_agent tool loop)
# ============================================================================
def run_search_agent(query: str) -> str:
    """Part 1's Researcher agent: a create_agent loop that runs Tavily searches, for live or external information."""
    console.print("--- STRATEGY [search_agent]: Running the create_agent search loop... ---")
    answer, tool_calls = ask(SPECIALIST_AGENTS["Researcher"], query)
    console.print(f"    [searches] {tool_calls} tool call(s)")
    return answer


print("✅ run_search_agent() defined: reuses Part 1's Researcher agent")

In [ ]:
# ============================================================================
# STRATEGY 3: MULTI-STEP PLANNING
# ============================================================================
class SubSteps(BaseModel):
    """An ordered list of sub-questions that together cover the full task."""
    steps: List[str] = Field(description="Ordered list of self-contained sub-questions.")


def run_multi_step_planning(query: str) -> str:
    """Decompose -> answer each sub-step -> synthesize.

    For multi-part questions that bundle several distinct sub-questions together.
    """
    console.print("--- STRATEGY [multi_step_planning]: Decomposing into sub-steps... ---")
    planner_llm = llm.with_structured_output(SubSteps)
    plan = planner_llm.invoke(
        f"Break the following task into an ordered list of self-contained sub-questions "
        f"that together fully answer it.\n\nTask: {query}"
    )
    steps = plan.steps[:MAX_SUB_STEPS]  # cap: every sub-question is one more LLM call
    console.print(f"    [plan] {steps} ({len(plan.steps)} proposed, at most {MAX_SUB_STEPS} answered)")

    answers = []
    for step in steps:
        answer = llm.invoke(f"Answer this sub-question concisely: {step}").content
        console.print(f"    [sub-step] {step} -> {answer[:80]}...")
        answers.append(f"Sub-question: {step}\nAnswer: {answer}")

    context = "\n\n".join(answers)
    final = llm.invoke(
        f"""Combine the sub-answers below into one coherent final answer to the original task.

Original task: {query}

Sub-answers:
{context}"""
    )
    return final.content

print(f"✅ run_multi_step_planning() defined: at most {MAX_SUB_STEPS + 2} LLM calls")

In [ ]:
# ============================================================================
# STRATEGY 4: DEBATE
# ============================================================================
def run_debate(query: str) -> str:
    """Two opposing stances, then a judge synthesizes a balanced verdict.

    For opinion / trade-off questions that don't have one factual answer.
    """
    console.print("--- STRATEGY [debate]: Generating opposing stances... ---")
    for_case = llm.invoke(
        f"Argue IN FAVOR of the following, as persuasively and honestly as possible, "
        f"in 3-4 sentences.\n\nTopic: {query}"
    ).content
    against_case = llm.invoke(
        f"Argue AGAINST the following, as persuasively and honestly as possible, "
        f"in 3-4 sentences.\n\nTopic: {query}"
    ).content
    console.print("    [for] " + for_case[:80] + "...")
    console.print("    [against] " + against_case[:80] + "...")

    verdict = llm.invoke(
        f"""You are an impartial judge. Weigh the two stances below on the topic and give a
balanced verdict that acknowledges genuine trade-offs rather than declaring one side
simply "right".

Topic: {query}

FOR: {for_case}

AGAINST: {against_case}

Verdict:"""
    )
    return verdict.content

print("✅ Four strategies defined: direct_answer, search_agent, multi_step_planning, debate")


---
## 🧠 Part 3: The Classifier and the Registry

The meta-controller is a structured-output classifier plus a registry. `MetaControllerDecision.strategy` is a
`Strategy` Enum, so the model can only name a known architecture, and `STRATEGY_REGISTRY` maps each Enum member to
its callable instead of a chain of `if`s. `meta_controller()` classifies, then dispatches. The registry cell checks
in code that every `Strategy` member has a callable.

In [ ]:
# ============================================================================
# META-CONTROLLER SCHEMA
# ============================================================================
class Strategy(str, Enum):
    DIRECT_ANSWER = "direct_answer"
    SEARCH_AGENT = "search_agent"
    MULTI_STEP_PLANNING = "multi_step_planning"
    DEBATE = "debate"


class MetaControllerDecision(BaseModel):
    """The Meta-Controller's choice of which whole architecture should handle the task."""
    strategy: Strategy = Field(description="The single best-suited strategy for this task.")
    reasoning: str = Field(
        description="One to two sentences explaining why this strategy fits the task's shape."
    )


STRATEGY_DESCRIPTIONS = """\
- direct_answer: The task is a simple, self-contained factual question the model already
  knows (no live/external data needed, no decomposition needed, not a matter of opinion).
- search_agent: The task requires current, live, or external information (news, prices,
  recent events, anything that could have changed since training) that a web search can find.
- multi_step_planning: The task bundles multiple distinct sub-questions or steps that each
  need to be answered and then combined into one coherent final answer.
- debate: The task asks for an opinion, recommendation, or judgment about a trade-off with
  genuine merit on more than one side (no single factual answer exists).
"""

STRATEGY_REGISTRY = {
    Strategy.DIRECT_ANSWER: run_direct_answer,
    Strategy.SEARCH_AGENT: run_search_agent,
    Strategy.MULTI_STEP_PLANNING: run_multi_step_planning,
    Strategy.DEBATE: run_debate,
}

missing = set(Strategy) - set(STRATEGY_REGISTRY)
print(f"{'✅' if not missing else '❌'} {len(STRATEGY_REGISTRY)} strategies registered: {[s.value for s in STRATEGY_REGISTRY]}; "
      f"Strategy members without a callable: {[s.value for s in missing] or 'none'}")


In [ ]:
# ============================================================================
# META-CONTROLLER NODE
# ============================================================================
def meta_controller(query: str) -> dict:
    """Classify which agentic architecture best fits the task, then dispatch to it."""
    console.print(f"\n=== META-CONTROLLER: Classifying task ===\n'{query}'")

    classifier_llm = llm.with_structured_output(MetaControllerDecision)
    decision = classifier_llm.invoke(
        f"""You are a Meta-Controller. Your job is to choose which ONE agentic architecture
should handle the task below -- not to answer the task yourself.

Available strategies:
{STRATEGY_DESCRIPTIONS}

Task: {query}

Choose the single best-suited strategy and explain your reasoning."""
    )
    console.print(
        f"--- DECISION: strategy='{decision.strategy.value}' | reasoning: {decision.reasoning} ---"
    )

    strategy_fn = STRATEGY_REGISTRY[decision.strategy]
    output = strategy_fn(query)

    return {
        "query": query,
        "strategy": decision.strategy.value,
        "reasoning": decision.reasoning,
        "output": output,
    }

print("✅ meta_controller() ready: classify, then dispatch through STRATEGY_REGISTRY")

### 3.1 ▶️ Four Tasks, Four Shapes

The same controller, with the same classification prompt, gets four tasks, each shaped for a different strategy:

| Task | Shape | Expected strategy | Why the others are worse |
|---|---|---|---|
| Chemical symbol for gold | self-contained fact | `direct_answer` | a search loop or a debate adds calls without improving the answer |
| Price of Bitcoin today | live, changing value | `search_agent` | the model cannot know it from training data; a single call would guess |
| Largest planet: which, why so large, compared with Earth | three bundled sub-questions | `multi_step_planning` | one direct call risks shortchanging one of the parts |
| Fully remote work vs office attendance | trade-off, no single answer | `debate` | a single call tends to present one side as simply correct |

Once chosen, the search agent's own loop decides how many searches to run. That is a lower-level decision than the
meta-controller's choice of architecture. Classification can still go wrong (the *classification errors cascade*
weakness above); Part 4 checks every choice.

In [ ]:
# ============================================================================
# RUN FOUR TASKS THROUGH THE META-CONTROLLER
# ============================================================================
meta_cases = [
    ("What is the chemical symbol for gold?", "direct_answer"),
    ("What is the current price of Bitcoin in USD today?", "search_agent"),
    ("Name the largest planet in the solar system, explain why it is that large, and compare its size to Earth.",
     "multi_step_planning"),
    ("Should companies adopt a fully remote work policy instead of requiring office attendance?", "debate"),
]

meta_results = []
for task, expected in meta_cases:
    result = meta_controller(task)
    console.print("\n[bold]Final Output:[/bold]")
    console.print(Markdown(result["output"]))
    meta_results.append((task, expected, result["strategy"]))

---
## ✅ Part 4: One Route Check for Both Controllers

Both controllers are a single point of failure, so the cheapest regression test for either one is a list of requests
with the expected route next to each. The table covers the router's three requests and the meta-controller's four
tasks, computed from what each controller actually chose, plus one line on whether the Researcher really searched.

In [ ]:
# ============================================================================
# ROUTE CHECK: expected vs chosen, for both controllers
# ============================================================================
rows = [("router", request, expected, chosen) for request, expected, chosen, _ in router_results]
rows += [("meta-controller", task, expected, chosen) for task, expected, chosen in meta_results]

print(f"   {'controller':<16} {'expected':<20} {'chosen':<20} request")
for controller, request, expected, chosen in rows:
    print(f"{'✅' if chosen == expected else '❌'} {controller:<16} {expected:<20} {chosen:<20} {request[:45]}")

print()
for controller in ("router", "meta-controller"):
    mine = [row for row in rows if row[0] == controller]
    print(f"{controller}: {sum(row[2] == row[3] for row in mine)}/{len(mine)} routes as expected")

searches = [calls for _, _, chosen, calls in router_results if chosen == "Researcher"]
print(f"{'✅' if searches and min(searches) > 0 else '❌'} the Researcher ran its search tool (tool calls: {searches})")

**Reading the output:** a ✅ row means the controller chose the route that request was written for. A ❌ row is the
*classification errors cascade* weakness happening live: the request still gets an answer, from the wrong specialist
or architecture, and nothing downstream notices. That is why the check is computed rather than eyeballed, and why a
table like this belongs in a regression test.

---
## 📝 Summary

In this notebook, we learned:

### 1. Routing as a graph
- A router is one typed decision: a `Literal` field can only name a real node, and a pure routing function reads it from state
- Tool-using specialists must run their tools (`create_agent`); `bind_tools()` alone does not
- Unlike a handoff, the router decides before any specialist runs, and the specialist cannot pass the request on

### 2. Strategies as callables
- Whole architectures sit behind one `(query) -> str` signature; the search strategy is Part 1's Researcher agent, reused
- Each strategy keeps its own control flow (a tool loop, a plan, a debate), invisible to the controller

### 3. Classifier and registry
- A `Strategy` Enum decision plus `STRATEGY_REGISTRY` replace a chain of `if`s; a new strategy is one callable and one description line, and the registry check catches a missing callable

### 4. Checked routing
- One code-computed table of expected vs chosen routes is the cheapest regression test for both controllers

### Router, blackboard controller and meta-controller

| Pattern | What it decides | When it decides | Where |
|---|---|---|---|
| Router | which specialist (or tool) handles the request, inside one fixed graph | once, before the specialist runs | Part 1; module 06's supervisor without the loop |
| Blackboard controller | which specialist acts next, given everything on the shared board | after every contribution, until the task is done | `01_blackboard_intro.ipynb`, `02_blackboard_research_team.ipynb` |
| Meta-controller | which whole architecture (single call, search agent, planner, debate) to run | once, before any architecture runs | Parts 2–3 |

### Next Steps
- **`06_ensemble_parallel_exploration.ipynb`**: several independent agents work on the same task at once, and an aggregator combines their views